In [ ]:
import pandas as pd
import os

In [ ]:

# Agora ele vai listar os arquivos que estão "ao lado" dele
print(os.listdir('.')) 

# Carregando sem o prefixo da pasta
df_customers = pd.read_csv('olist_customers_dataset.csv')
df_orders = pd.read_csv('olist_orders_dataset.csv')

# Visualizando
df_customers.head()

In [ ]:
# 1. Carregar a tabela de itens (que tem os preços)
df_items = pd.read_csv('olist_order_items_dataset.csv')

# 2. Unir Clientes com Pedidos (para ter o unique_id vinculado ao pedido)
# Usamos 'customer_id' como a chave de ligação
df_temp = pd.merge(df_customers, df_orders, on='customer_id')

# 3. Unir o resultado com os Itens (para ter o preço vinculado ao cliente)
# Usamos 'order_id' como a chave de ligação
df_final = pd.merge(df_temp, df_items, on='order_id')

# Ver as colunas que sobraram e as primeiras linhas
print(df_final.columns)
df_final[['customer_unique_id', 'price', 'customer_city']].head()

In [ ]:
# 1. Agregando por SUM e MEAN ao mesmo tempo
relatorio_vendas = df_final.groupby('customer_state')['price'].agg(['sum', 'mean', 'count']).reset_index()

# 2. Renomeando as colunas para ficar profissional e fácil de entender
relatorio_vendas.columns = ['Estado', 'Faturamento_Total', 'Ticket_Medio', 'Quantidade_Pedidos']

# 3. Ordenando pelo faturamento total para manter a lógica anterior
relatorio_vendas = relatorio_vendas.sort_values(by='Faturamento_Total', ascending=False)

# 4. Criando a coluna de percentual (usando o faturamento total)
total_geral = relatorio_vendas['Faturamento_Total'].sum()
relatorio_vendas['%_do_Total'] = (relatorio_vendas['Faturamento_Total'] / total_geral) * 100

print(relatorio_vendas.head(10))

In [ ]:
# Formatando para ficar legível
relatorio_formatado = relatorio_vendas.copy()
relatorio_formatado['Faturamento_Total'] = relatorio_formatado['Faturamento_Total'].map('R$ {:,.2f}'.format)
relatorio_formatado['Ticket_Medio'] = relatorio_formatado['Ticket_Medio'].map('R$ {:,.2f}'.format)
relatorio_formatado['%_do_Total'] = relatorio_formatado['%_do_Total'].map('{:.2f}%'.format)

print(relatorio_formatado.head(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Configurando o estilo do gráfico
plt.figure(figsize=(12,6))
sns.barplot(data=faturamento_estado.head(10), x='customer_state', y='price', palette='viridis')

# Adicionando títulos e rótulos
plt.title('Top 10 Estados por Faturamento Total (Olist)', fontsize=15)
plt.xlabel('Estado', fontsize=12)
plt.ylabel('Faturamento em Milhões (R$)', fontsize=12)

# Exibindo o gráfico
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
ax = sns.barplot(data=relatorio_vendas.head(10), x='Estado', y='Faturamento_Total', palette='viridis')

# Adicionando os números em cima das barras
for container in ax.containers:
    ax.bar_label(container, fmt='R$ {:,.0f}', padding=3)

plt.title('Top 10 Estados por Faturamento Total', fontsize=15)
plt.show()

In [ ]:
fig, ax1 = plt.subplots(figsize=(14,7))

# Gráfico de Barras (Faturamento)
sns.barplot(data=relatorio_vendas.head(10), x='Estado', y='Faturamento_Total', alpha=0.6, ax=ax1, palette='Blues_r')
ax1.set_ylabel('Faturamento Total (R$)', color='b', fontsize=12)

# Criando um segundo eixo para a Linha (Ticket Médio)
ax2 = ax1.twinx()
sns.lineplot(data=relatorio_vendas.head(10), x='Estado', y='Ticket_Medio', marker='o', color='red', linewidth=3, ax=ax2)
ax2.set_ylabel('Ticket Médio (R$)', color='r', fontsize=12)

plt.title('Volume de Faturamento vs Ticket Médio por Estado', fontsize=16)
plt.show()

In [ ]:
# 1. Carregar a tabela de reviews
df_reviews = pd.read_csv('olist_order_reviews_dataset.csv')

# 2. Unir com o nosso dataframe principal (df_final)
# Usamos 'left' para não perder pedidos que por acaso não tenham review
df_analise_satisfacao = pd.merge(df_final, df_reviews, on='order_id', how='left')

# 3. Criar o ranking de satisfação por estado
satisfacao_estado = df_analise_satisfacao.groupby('customer_state')['review_score'].mean().sort_values(ascending=False).reset_index()

# 4. Mostrar os 5 estados mais felizes e os 5 mais insatisfeitos
print("Estados com as MELHORES notas médias:")
print(satisfacao_estado.head(5))

print("\nEstados com as PIORES notas médias:")
print(satisfacao_estado.tail(5))

In [ ]:
# 1. Convertando as colunas de data para o formato datetime
df_final['order_delivered_customer_date'] = pd.to_datetime(df_final['order_delivered_customer_date'])
df_final['order_estimated_delivery_date'] = pd.to_datetime(df_final['order_estimated_delivery_date'])

# 2. Calculando a diferença em dias entre a data de entrega e a data estimada
# .dt.days extrai apenas o numero de dias da diferença
df_final['atraso_entrega'] = (df_final['order_delivered_customer_date'] - df_final['order_estimated_delivery_date']).dt.days

# 3. Criando relatório de atraso médio por estado
atraso_estado = df_final.groupby('customer_state')['atraso_entrega'].mean().sort_values(ascending=False).reset_index()
# Renomeando as colunas para ficar mais claro
atraso_estado.columns = ['Estado', 'Atraso_Medio']
print("Estados com maior atraso médio na entrega:")
print(atraso_estado.head(10))

print("Estados com menor atraso médio na entrega:")
print(atraso_estado.tail(10))



In [ ]:
plt.figure(figsize=(14,6))
sns.barplot(data=atraso_estado, x='Estado', y='Atraso_Medio', palette='RdYlGn_r')
plt.axhline(0, color='black', linestyle='--') # Linha do "Prazo Cumprido"
plt.title('Atraso Médio na Entrega por Estado (Dias)', fontsize=15)
plt.ylabel('Dias (Positivo = Atraso | Negativo = Adiantado)')
plt.show()

In [ ]:
# Vamos carregar a tabela de produtos e a tradução de categorias para ter um relatório mais completo de quais tipos de produtos estão vendendo mais e gerando mais receita.
df_products = pd.read_csv('olist_products_dataset.csv')
df_category = pd.read_csv('product_category_name_translation.csv')

# 2. Unir os produtos com a tradução para termos nomes em inglês (opcional, mas profissional)
df_products = pd.merge(df_products, df_category, on='product_category_name', how='left')

# 3. Unir com o nosso df_final para saber quais produtos foram vendidos
df_analise_produtos = pd.merge(df_final, df_products, on='product_id', how='left')

# 4. Criar o ranking por Categoria (Quantidade e Faturamento)
categoria_vendas = df_analise_produtos.groupby('product_category_name_english').agg({'price': ['sum', 'count']}).reset_index()
categoria_vendas.columns = ['Categoria', 'Faturamento_Total', 'Quantidade_Vendida']

# 5. Ver o top 10 categorias por Produto
top_10_produtos = categoria_vendas.sort_values(by='Quantidade_Vendida', ascending=False).head(10)
print("Top 10 Categorias mais vendidas (Quantidade):")
print(top_10_produtos)

In [ ]:
# Vamos criar um gráfico de barras horizontal para mostrar as categorias mais vendidas em quantidade, e um gráfico de pizza para mostrar a participação de cada categoria no faturamento total.
plt.figure(figsize=(12,6))
sns.barplot(data=top_10_produtos, x='Quantidade_Vendida', y='Categoria', palette='viridis')
plt.title('Top 10 Categorias por Quantidade Vendida', fontsize=15)

In [ ]:
import plotly.express as px

# Criando um gráfico interativo de Treemap
fig = px.treemap(top_10_produtos, 
                 path=['Categoria'], 
                 values='Quantidade_Vendida',
                 color='Faturamento_Total', 
                 color_continuous_scale='Viridis',
                 title='Distribuição de Vendas: Volume vs Faturamento')

fig.show()

In [44]:
# Salva o dataframe principal limpo para usar no Power BI
df_final.to_csv('olist_dados_limpos.csv', index=False)